# 09 — Paper figure: pixel‑intensity histograms per exposure and HDR

Fig. 8 of the paper: normalised pixel‑intensity distributions of the nine
stacked exposures and of the two HDR products, per polariser, with the number
of frames actually stacked annotated per column.

Two layouts are saved: `histogram_normalized_exposures_vertical.{png,pdf}` (the
paper's: intensity on y, counts on x) and `..._horizontal.{png,pdf}` (conventional:
intensity on x, log counts on y, 4 × 3 grid).
Legacy source: `statistical_analysis_figures.ipynb` cell 11.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
from scipy import ndimage
table = pd.read_csv(config.FRAME_TABLE_CSV, dtype={"position": str})
good = table[table.use]          # frames actually stacked (radius filter + config.EXTRA_EXCLUDED_FRAMES)
counts = {pos: {e: int(((good.position == pos) & (good.inverse_exposure_time == e)).sum()) for e in config.INV_EXPOSURES}
          for pos in config.POLARIZER_POSITIONS}

def norm01(img):
    v = img[img > 0].ravel()
    return v / v.max()

hists = {pos: [] for pos in config.POLARIZER_POSITIONS}
for e in config.INV_EXPOSURES:
    d = fits.getdata(utils.stacked_filename(e))
    for i, pos in enumerate(config.POLARIZER_POSITIONS):
        plane = np.asarray(d[i], np.float32)
        if any(config.CHANNEL_SHIFTS[pos]):
            plane = ndimage.shift(plane, config.CHANNEL_SHIFTS[pos], order=1)
        hists[pos].append(norm01(plane))
for method in ("ldic", "expnorm"):
    for pos in config.POLARIZER_POSITIONS:
        hists[pos].append(norm01(fits.getdata(utils.hdr_filename(method, pos))))

In [ ]:
# Paper layout (vertical): intensity on y, log counts on x, 11 panels side by side
fig = utils.plot_exposure_histograms(hists, counts, orientation="vertical",
                                     out_stem=config.FIGURES_DIR / "histogram_normalized_exposures_vertical")

In [ ]:
# Conventional layout (horizontal): intensity on x, log counts on y, 4 x 3 grid
fig = utils.plot_exposure_histograms(hists, counts, orientation="horizontal",
                                     out_stem=config.FIGURES_DIR / "histogram_normalized_exposures_horizontal")